In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
from abc import ABC, abstractmethod
import numpy as np
from sklearn.metrics import classification_report, r2_score, mean_absolute_error, root_mean_squared_error, mean_squared_error
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import tensorflow as tf
import optuna


In [3]:
df = pd.read_csv('diamonds.csv')
print(df.columns.tolist())

data = pd.get_dummies(df, columns=['cut','color','clarity'])

data = data.dropna(axis=1, how='all')  
data.fillna(data.mean(numeric_only=True), inplace=True)
X = data.drop(['price'], axis=1).values

y = data['price'].values


print(f"Признаков: {X.shape[1]}, Объектов: {X.shape[0]}")



X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

scaler_y = StandardScaler()
y_train = scaler_y.fit_transform(y_train.reshape(-1, 1)).ravel()
y_test = scaler_y.transform(y_test.reshape(-1, 1)).ravel()

['carat', 'cut', 'color', 'clarity', 'depth', 'table', 'x', 'y', 'z', 'price']
Признаков: 26, Объектов: 53940


In [5]:
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1) 
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=64, shuffle=False)


class DiamondRegression(nn.Module):
    def __init__(self, input_size):
        super(DiamondRegression, self).__init__()
        self.fc1 = nn.Linear(input_size, 8)
        self.fc2 = nn.Linear(8, 4)
        self.fc3 = nn.Linear(4, 1)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

model = DiamondRegression(X.shape[1])

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


for epoch in range(15):
    model.train()
    running_loss = 0.0
    
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    if (epoch + 1) % 5 == 0:
        print(f"Эпоха {epoch+1}/15, Потери: {running_loss/len(train_loader):.4f}")


model.eval()
y_pred_scaled = []
y_true_scaled = []

with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = model(inputs)
        y_pred_scaled.extend(outputs.numpy().flatten())
        y_true_scaled.extend(labels.numpy().flatten())

y_pred = scaler_y.inverse_transform(np.array(y_pred_scaled).reshape(-1, 1)).flatten()
y_true = scaler_y.inverse_transform(np.array(y_true_scaled).reshape(-1, 1)).flatten()

print(f"MSE: {mean_squared_error(y_true, y_pred):.2f}")
print(f"RMSE: {root_mean_squared_error(y_true, y_pred):.2f}")
print(f"MAE: {mean_absolute_error(y_true, y_pred):.2f}")
print(f"R²: {r2_score(y_true, y_pred):.4f}")

Эпоха 5/15, Потери: 0.0315
Эпоха 10/15, Потери: 0.0246
Эпоха 15/15, Потери: 0.0234
MSE: 363912.72
RMSE: 603.25
MAE: 341.58
R²: 0.9771


Подбор гиперпараметров с Optuna

In [ ]:
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1) 
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

def objective(trial):
    n_layers = trial.suggest_int('n_layers', 1, 4)
    hidden_sizes = []
    for i in range(n_layers):
        hidden_sizes.append(trial.suggest_int(f'hidden_size_{i}', 8, 128, step=8))
    
    dropout = trial.suggest_float('dropout', 0.0, 0.5)
    lr = trial.suggest_float('lr', 1e-5, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch_size', [32, 64, 128])
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)
    epochs = trial.suggest_int('epochs', 20, 80, step=10)
    
    
    class DiamondRegression(nn.Module):
        def __init__(self):
            super().__init__()
            layers = []
            prev = X_train.shape[1]
            for h in hidden_sizes:
                layers.append(nn.Linear(prev, h))
                layers.append(nn.BatchNorm1d(h))
                layers.append(nn.ReLU())
                layers.append(nn.Dropout(dropout))
                prev = h
            layers.append(nn.Linear(prev, 1))
            self.net = nn.Sequential(*layers)
        
        def forward(self, x):
            return self.net(x)
    
    model = DiamondRegression()
    
    train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=batch_size, shuffle=False)
    
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.MSELoss()
    
    best_val_loss = float('inf')
    patience = 0
    
    for epoch in range(epochs):
        model.train()
        for inputs, labels in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(inputs), labels)
            loss.backward()
            optimizer.step()
        
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                val_loss += criterion(model(inputs), labels).item()
        val_loss /= len(val_loader)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience = 0
        else:
            patience += 1
            if patience >= 5:
                break
    
    return best_val_loss

study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=30)

best_params = study.best_params
print("Best params:", best_params)

n_layers = best_params['n_layers']
hidden_sizes = [best_params[f'hidden_size_{i}'] for i in range(n_layers)]

class FinalModel(nn.Module):
    def __init__(self):
        super().__init__()
        layers = []
        prev = X_train.shape[1]
        for h in hidden_sizes:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(best_params['dropout']))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x)

model = FinalModel()
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=best_params['batch_size'], shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=best_params['batch_size'], shuffle=False)

optimizer = optim.Adam(model.parameters(), lr=best_params['lr'], weight_decay=best_params['weight_decay'])
criterion = nn.MSELoss()

best_loss = float('inf')
patience = 0

for epoch in range(best_params['epochs']):
    model.train()
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(inputs), labels)
        loss.backward()
        optimizer.step()
    
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, labels in test_loader:
            val_loss += criterion(model(inputs), labels).item()
    val_loss /= len(test_loader)
    
    if val_loss < best_loss:
        best_loss = val_loss
        patience = 0
        torch.save(model.state_dict(), 'best_model.pth')
    else:
        patience += 1
        if patience >= 7:
            break

model.load_state_dict(torch.load('best_model.pth'))
model.eval()

y_pred_scaled = []
y_true_scaled = []

with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = model(inputs)
        y_pred_scaled.extend(outputs.numpy().flatten())
        y_true_scaled.extend(labels.numpy().flatten())

y_pred = scaler_y.inverse_transform(np.array(y_pred_scaled).reshape(-1, 1)).flatten()
y_true = scaler_y.inverse_transform(np.array(y_true_scaled).reshape(-1, 1)).flatten()

print(f"MSE: {mean_squared_error(y_true, y_pred):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_true, y_pred)):.2f}")
print(f"MAE: {mean_absolute_error(y_true, y_pred):.2f}")
print(f"R²: {r2_score(y_true, y_pred):.4f}")

[I 2026-08-20 14:29:02,468] A new study created in memory with name: no-name-7ff752ad-58e7-41c7-92ff-30e4f54725da
[I 2026-08-20 14:29:33,796] Trial 0 finished with value: 0.03850828154998667 and parameters: {'n_layers': 2, 'hidden_size_0': 128, 'hidden_size_1': 96, 'dropout': 0.2993292420985183, 'lr': 2.9380279387035334e-05, 'batch_size': 128, 'weight_decay': 6.358358856676247e-05, 'epochs': 60}. Best is trial 0 with value: 0.03850828154998667.
[I 2026-08-20 14:30:21,145] Trial 1 finished with value: 0.031698444836279926 and parameters: {'n_layers': 1, 'hidden_size_0': 128, 'dropout': 0.41622132040021087, 'lr': 4.335281794951564e-05, 'batch_size': 128, 'weight_decay': 3.752055855124284e-05, 'epochs': 50}. Best is trial 1 with value: 0.031698444836279926.
[I 2026-08-20 14:30:55,661] Trial 2 finished with value: 0.02660405893462914 and parameters: {'n_layers': 2, 'hidden_size_0': 80, 'hidden_size_1': 24, 'dropout': 0.14607232426760908, 'lr': 0.00012562773503807024, 'batch_size': 64, 'wei

Best params: {'n_layers': 3, 'hidden_size_0': 32, 'hidden_size_1': 64, 'hidden_size_2': 56, 'dropout': 0.013283975193522715, 'lr': 0.0014252280669512017, 'batch_size': 128, 'weight_decay': 7.235683548981251e-06, 'epochs': 40}
MSE: 322177.03
RMSE: 567.61
MAE: 312.98
R²: 0.9797


In [ ]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(X.shape[1],)), 
    
    tf.keras.layers.Dense(8, activation='relu'), 
    tf.keras.layers.Dense(4, activation='relu'), 
    
    tf.keras.layers.Dense(1) 
])


model.compile(
    optimizer='adam',
    loss='mse',       
    metrics=['mae', 'mse']    
)

model.fit(X_train, y_train, epochs=20)
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
print(f"R²: {r2:.4f}")

Epoch 1/20
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 2s 920us/step - loss: 0.0666 - mae: 0.1422 - mse: 0.0666
Epoch 2/20
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 1s 923us/step - loss: 0.0285 - mae: 0.0971 - mse: 0.0285
Epoch 3/20
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 1s 923us/step - loss: 0.0253 - mae: 0.0910 - mse: 0.0253
Epoch 4/20
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 1s 938us/step - loss: 0.0245 - mae: 0.0888 - mse: 0.0245
Epoch 5/20
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 1s 935us/step - loss: 0.0237 - mae: 0.0872 - mse: 0.0237
Epoch 6/20
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 1s 945us/step - loss: 0.0237 - mae: 0.0860 - mse: 0.0237
Epoch 7/20
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 1s 909us/step - loss: 0.0231 - mae: 0.0846 - mse: 0.0231
Epoch 8/20
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 1s 916us/step - loss: 0.0228 - mae: 0.0838 - mse: 0.0228
Epoch 9/20
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 1s 917us/step - loss: 0.0219 - mae: 0.0830 - mse: 0.0219
Epoch 10/20
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 1s 910us/step - loss: 0.0221 - mae: 0.0825 - mse: 0.0221
Epoch 11/